# LABORATORIO 2 — Recolección de datos y aterrizaje en zona bronce

**Curso:** Analítica de Datos (202W1001 — Plan 2023)

**Escuela:** E.P. Ingeniería de Software — Facultad de Ingeniería de Sistemas e Informática — UNMSM

**Estudiante:** Luis Calle Huamantinco

**Código:** 22200255

**Semestre académico:** 2026-2 | Ciclo X | Unidad I

**Fecha:** 30/08/2026

**Docente:** Mg. Juan Gamarra Moreno

## Paso 0 - Configuracion del entorno

In [38]:
!pip install -q polars duckdb pyarrow requests pandas

In [39]:
import polars as pl
import duckdb
import pyarrow
import requests
import pandas as pd
print('polars :', pl.__version__)
print('duckdb :', duckdb.__version__)
print('pyarrow :', pyarrow.__version__)

polars : 1.44.1
duckdb : 1.5.5
pyarrow : 25.0.1


## Paso 1 — Crear la estructura de zonas

In [40]:
import os, json, time, hashlib
from datetime import datetime, date, timedelta
from pathlib import Path
import requests
import polars as pl
import duckdb
# Rutas del laboratorio
DATOS = Path('datos') # ajuste esta ruta si usa Google Drive
BASE = Path('lakehouse')
BRONCE = BASE / 'bronce' # dato crudo, inmutable
PLATA = BASE / 'plata' # dato validado y normalizado
CUARENTENA = BASE / 'cuarentena' # registros que no pasan el contrato
for zona in (BRONCE, PLATA, CUARENTENA):
 zona.mkdir(parents=True, exist_ok=True)
print('Zonas creadas:')
for zona in (BRONCE, PLATA, CUARENTENA):
 print(' ', zona)
print('\nArchivos fuente encontrados:')
for archivo in sorted(DATOS.glob('*')):
 print(f' {archivo.name:26s} {archivo.stat().st_size/1e6:7.2f} MB')


Zonas creadas:
  lakehouse/bronce
  lakehouse/plata
  lakehouse/cuarentena

Archivos fuente encontrados:
 bcrp_tipo_cambio.csv          0.00 MB
 clima_diario.jsonl            0.47 MB
 locales.csv                   0.00 MB
 maestro_productos.csv         0.03 MB
 ventas_pos.csv               10.54 MB


**Punto de control 1**

Sin problemas con los archivos

## Paso 2 — Reconocer la fuente antes de ingerirla

In [41]:
# a) Vistazo rápido: solo las primeras 1000 filas
vistazo = pl.read_csv(DATOS / 'ventas_pos.csv', n_rows=1000)
print('Esquema inferido:')
print(vistazo.schema)
# b) Carga completa, pidiendo a Polars que interprete las fechas
ventas_raw = pl.read_csv(DATOS / 'ventas_pos.csv', try_parse_dates=True)
print(f'\nFilas: {ventas_raw.height:,} | Columnas: {ventas_raw.width}')
ventas_raw.head(5)

Esquema inferido:
Schema({'id_venta': Int64, 'fecha': String, 'id_local': Int64, 'id_producto': Int64, 'cantidad': Int64, 'importe': Float64, 'canal': String, 'medio_pago': String})

Filas: 200,600 | Columnas: 8


id_venta,fecha,id_local,id_producto,cantidad,importe,canal,medio_pago
i64,date,i64,i64,i64,f64,str,str
77288,2025-12-10,77,1203,2,30.06,"""Presencial""","""Billetera"""
129421,2025-09-09,84,1296,6,106.15,"""Delivery""","""Efectivo"""
196899,2025-10-26,74,1220,2,47.45,"""Presencial""","""Billetera"""
16250,2026-03-12,61,1092,3,229.1,"""Presencial""","""Tarjeta"""
23062,2026-02-01,73,1399,3,23.73,"""Presencial""","""Efectivo"""


In [42]:
perfil = pl.DataFrame({
 'columna': ventas_raw.columns,
 'tipo': [str(t) for t in ventas_raw.dtypes],
 'nulos': [ventas_raw[c].null_count() for c in ventas_raw.columns],
 'distintos': [ventas_raw[c].n_unique() for c in ventas_raw.columns],
})
perfil

columna,tipo,nulos,distintos
str,str,i64,i64
"""id_venta""","""Int64""",0,200000
"""fecha""","""Date""",0,366
"""id_local""","""Int64""",0,120
"""id_producto""","""Int64""",0,898
"""cantidad""","""Int64""",0,26
"""importe""","""Float64""",4012,39656
"""canal""","""String""",0,12
"""medio_pago""","""String""",0,3


**Punto de control 2**

**2.1. El archivo tiene 200 600 filas pero id_venta solo tiene 200 000 valores distintos. ¿Qué
explica la diferencia?**

Se tienen datos con el mismo id_venta, 600 de manera concreta.

**2.2. La columna canal tiene 12 valores distintos, pero solo existen tres canales de venta
reales. ¿Qué está ocurriendo?**

Puede que los nombres de los canales tengan variaciones como espacios o mayusculas por lo que se
cuentan como valores distintos.

**2.3. La columna importe tiene 4 010 nulos. ¿Debe eliminar esas filas, imputarlas o aislarlas?
Justifique.**

Imputarlas, eliminar estas filas puede afectar el estudio de los datos, el aislarlo no nos daria ningun valor ya que son datos nulos. Al imputarlas seguimos tomando en cuenta estas filas para un estudio adecuado de estos datos.

**2.4. id_producto tiene 898 valores distintos, pero el maestro de productos solo tiene 800
registros. ¿Qué implica esto?**

Que hay codigos de producto en las ventas que no estan registrados en el maestro de productos, o sea hay ventas con productos que no existen en el catalogo (huerfanos).

## Paso 3 — Aterrizar en la zona bronce con procedencia

In [43]:
def huella(ruta, bloque=1 << 20):
 """Devuelve los primeros 16 caracteres del SHA-256 del archivo.
 Permite detectar si dos ejecuciones leyeron exactamente el mismo
 contenido de origen, lo que es la base para verificar idempotencia.
 """
 h = hashlib.sha256()
 with open(ruta, 'rb') as fh:
    for trozo in iter(lambda: fh.read(bloque), b''):
        h.update(trozo)
 return h.hexdigest()[:16]

In [44]:
def aterrizar_bronce(df, fuente, origen, particion=None):
 """Escribe un DataFrame en la zona bronce y registra la carga."""
 particion = particion or date.today().isoformat()
 destino = BRONCE / fuente / f'fecha_carga={particion}'
 destino.mkdir(parents=True, exist_ok=True)
 
 obtenido = datetime.now().isoformat(timespec='seconds')
 # Columnas de procedencia: acompañan a cada fila por todo el pipeline
 df = df.with_columns([
    pl.lit(fuente).alias('_fuente'),
    pl.lit(str(origen)).alias('_origen'),
    pl.lit(obtenido).alias('_obtenido_en'),
 ])

 archivo = destino / f'{fuente}.parquet'
 df.write_parquet(archivo, compression='zstd')

 # Manifiesto: una línea JSON por cada carga realizada
 registro = {
    'fuente': fuente,
    'origen': str(origen),
    'particion': particion,
    'obtenido_en': obtenido,
    'filas': df.height,
    'columnas': df.width,
    'archivo': str(archivo),
    'huella_origen': huella(origen) if Path(origen).is_file() else None,
 }
 with open(BRONCE / '_manifiesto.jsonl', 'a', encoding='utf-8') as fh:
    fh.write(json.dumps(registro, ensure_ascii=False) + '\n')

 print(f' bronce <- {fuente}: {df.height:,} filas en {archivo}')
 return archivo

In [45]:
aterrizar_bronce(ventas_raw, 'ventas_pos', DATOS / 'ventas_pos.csv')

aterrizar_bronce(pl.read_csv(DATOS / 'maestro_productos.csv'), 'maestro_productos', DATOS / 'maestro_productos.csv')

aterrizar_bronce(pl.read_csv(DATOS / 'locales.csv'), 'locales', DATOS / 'locales.csv')

# Revisar el manifiesto
print('\n--- Manifiesto de cargas ---')

for linea in open(BRONCE / '_manifiesto.jsonl', encoding='utf-8'):
    reg = json.loads(linea)
    print(f"{reg['fuente']:20s} {reg['filas']:>8,} filas "
        f"huella={reg['huella_origen']}")

 bronce <- ventas_pos: 200,600 filas en lakehouse/bronce/ventas_pos/fecha_carga=2026-09-01/ventas_pos.parquet
 bronce <- maestro_productos: 800 filas en lakehouse/bronce/maestro_productos/fecha_carga=2026-09-01/maestro_productos.parquet
 bronce <- locales: 120 filas en lakehouse/bronce/locales/fecha_carga=2026-09-01/locales.parquet

--- Manifiesto de cargas ---
ventas_pos            200,600 filas huella=9fd542c66e519295
maestro_productos         800 filas huella=9801f47eafce8b3e
locales                   120 filas huella=88087d8703f28356
ventas_pos            200,600 filas huella=9fd542c66e519295
maestro_productos         800 filas huella=9801f47eafce8b3e
locales                   120 filas huella=88087d8703f28356
ventas_pos            200,600 filas huella=9fd542c66e519295
maestro_productos         800 filas huella=9801f47eafce8b3e
locales                   120 filas huella=88087d8703f28356
clima_diario            2,920 filas huella=8c148f1131bf362d
bcrp_tipo_cambio          261 filas 

**Punto de control 3**

Como vemos, la huella es la misma sin importar el momento de ejecucion. Esto nos dice que nuestro proceso es idempotente, ya que estamos leyendo el mismo archivo origen.

## Paso 4 — Ingerir una fuente semiestructurada y detectar deriva de esquema

In [46]:
brutos = [json.loads(linea)
          for linea in open(DATOS / 'clima_diario.jsonl', encoding='utf-8')]

print('Registros leídos:', len(brutos))
print('\nPRIMER registro:')
print(json.dumps(brutos[0], indent=2, ensure_ascii=False))
print('\nÚLTIMO registro:')
print(json.dumps(brutos[-1], indent=2, ensure_ascii=False))

Registros leídos: 2920

PRIMER registro:
{
  "estacion": {
    "region": "Lima",
    "codigo": "EST-01"
  },
  "fecha": "2025-09-01",
  "medicion": {
    "humedad_pct": 59,
    "precipitacion_mm": 0.0,
    "temperatura_c": 15.0
  }
}

ÚLTIMO registro:
{
  "estacion": {
    "region": "Áncash",
    "codigo": "EST-08"
  },
  "fecha": "2026-08-31",
  "medicion": {
    "humedad_pct": 71,
    "precipitacion_mm": 0.4,
    "temp_c": 14.4,
    "indice_uv": 9
  }
}


In [47]:
conteo = {}
for reg in brutos:
    for clave in reg['medicion']:
        conteo[clave] = conteo.get(clave, 0) + 1

print('Presencia de cada campo dentro de medicion:')
for clave, n in sorted(conteo.items()):
    print(f' {clave:20s} {n:5d} de {len(brutos)} registros')

Presencia de cada campo dentro de medicion:
 humedad_pct           2920 de 2920 registros
 indice_uv              736 de 2920 registros
 precipitacion_mm      2920 de 2920 registros
 temp_c                 736 de 2920 registros
 temperatura_c         2184 de 2920 registros


In [48]:
filas = []
for reg in brutos:
    m = reg['medicion']
    filas.append({
        'region': reg['estacion']['region'],
        'codigo_estacion': reg['estacion']['codigo'],
        'fecha': reg['fecha'],
        'humedad_pct': m.get('humedad_pct'),
        'precipitacion_mm': m.get('precipitacion_mm'),
        # Resolución de la deriva: se acepta cualquiera de los dos nombres
        'temperatura_c': m.get('temperatura_c', m.get('temp_c')),
        'indice_uv': m.get('indice_uv'), # None antes de la deriva
        '_esquema': 'v2' if 'temp_c' in m else 'v1',
    })

# infer_schema_length=None obliga a Polars a revisar TODAS las filas antes
# de decidir los tipos. Sin esto, indice_uv (nulo en las primeras 2184
# filas) provoca un error de inferencia de tipo.
clima = (pl.DataFrame(filas, infer_schema_length=None)
         .with_columns(pl.col('fecha').str.to_date()))

print(clima.group_by('_esquema')
      .agg(pl.len().alias('registros'))
      .sort('_esquema'))
clima.head(3)

shape: (2, 2)
┌──────────┬───────────┐
│ _esquema ┆ registros │
│ ---      ┆ ---       │
│ str      ┆ u32       │
╞══════════╪═══════════╡
│ v1       ┆ 2184      │
│ v2       ┆ 736       │
└──────────┴───────────┘


region,codigo_estacion,fecha,humedad_pct,precipitacion_mm,temperatura_c,indice_uv,_esquema
str,str,date,i64,f64,f64,i64,str
"""Lima""","""EST-01""",2025-09-01,59,0.0,15.0,null,"""v1"""
"""Lima""","""EST-01""",2025-09-02,63,4.7,16.5,null,"""v1"""
"""Lima""","""EST-01""",2025-09-03,66,1.1,16.9,null,"""v1"""


In [49]:
aterrizar_bronce(clima, 'clima_diario', DATOS / 'clima_diario.jsonl')

 bronce <- clima_diario: 2,920 filas en lakehouse/bronce/clima_diario/fecha_carga=2026-09-01/clima_diario.parquet


PosixPath('lakehouse/bronce/clima_diario/fecha_carga=2026-09-01/clima_diario.parquet')

**Punto de control 4**

**El DataFrame clima tiene 2 920 filas, la columna temperatura_c no tiene ningún nulo, y la columna
indice_uv tiene exactamente 2 184 nulos. En una celda de texto, responda: si usted fuera el
consumidor de esta fuente, ¿qué cláusula habría incluido en el contrato de datos para que este
cambio no lo hubiera tomado por sorpresa?**

Para evitar este tipo de sorpresas en la clausula se debe especificar que no se puede hacer un
renombre o cambio en la estructura del json sin aviso o coordinacion con el consumidor.



## Paso 5 — Consumir una API pública con reintentos

In [50]:
def obtener_json(url, intentos=3, espera_base=2.0, timeout=20):
    """Cliente HTTP con reintentos y retroceso exponencial."""
    ultimo_error = None
    for n in range(1, intentos + 1):
        try:
            t0 = time.time()
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            meta = {
                'url': resp.url,
                'estado_http': resp.status_code,
                'segundos': round(time.time() - t0, 2),
                'intento': n,
            }
            return resp.json(), meta
        except requests.RequestException as e:
            ultimo_error = e
            print(f' intento {n} fallido: {e.__class__.__name__}')
            if n < intentos:
                espera = espera_base ** n
                print(f' reintentando en {espera:.1f} s...')
                time.sleep(espera)
    raise RuntimeError(f'Sin respuesta de {url}') from ultimo_error

In [51]:
SERIE = 'PN01207PD' # tipo de cambio bancario promedio, venta, diario
desde, hasta = date(2025, 9, 1), date(2026, 8, 31)

url = (f'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/'
    f'{SERIE}/json/{desde:%Y-%m-%d}/{hasta:%Y-%m-%d}/ing')
try:
    payload, meta = obtener_json(url)
    tipo_cambio = pl.DataFrame([
        {'periodo': p['name'], 'valor': p['values'][0]}
        for p in payload['periods']
    ])

    print('API en vivo OK:', tipo_cambio.height, 'períodos')
    print('Metadatos:', json.dumps(meta, indent=2, ensure_ascii=False))

    origen_tc = 'API BCRP en vivo'
except Exception as e:
    print(f'API no disponible ({type(e).__name__}). Se usa el respaldo local.')
    tipo_cambio = pl.read_csv(DATOS / 'bcrp_tipo_cambio.csv',
                              try_parse_dates=True)
    origen_tc = 'respaldo local'

print(f'\nOrigen efectivo: {origen_tc}')
tipo_cambio.head(5)

 intento 1 fallido: JSONDecodeError
 reintentando en 2.0 s...
 intento 2 fallido: JSONDecodeError
 reintentando en 4.0 s...
 intento 3 fallido: JSONDecodeError
API no disponible (RuntimeError). Se usa el respaldo local.

Origen efectivo: respaldo local


fecha,tipo_cambio_venta
date,f64
2025-09-01,3.7175
2025-09-02,3.7241
2025-09-03,3.7281
2025-09-04,3.7257
2025-09-05,3.7261


In [52]:
aterrizar_bronce(tipo_cambio, 'bcrp_tipo_cambio',
    DATOS / 'bcrp_tipo_cambio.csv')

 bronce <- bcrp_tipo_cambio: 261 filas en lakehouse/bronce/bcrp_tipo_cambio/fecha_carga=2026-09-01/bcrp_tipo_cambio.parquet


PosixPath('lakehouse/bronce/bcrp_tipo_cambio/fecha_carga=2026-09-01/bcrp_tipo_cambio.parquet')

**Punto de control 5**

**En una celda de texto, explique por qué la espera entre
reintentos crece de forma exponencial en lugar de ser constante, y qué problema causaría un
reintento inmediato en bucle contra un servicio saturado.**

Se realiza de forma exponencial porque si ha fallado por estar saturado o estar inactivo le damos mas tiempo de recomponerse a la API, si hicieramos un reintento inmediato, empeorariamos la saturacion al servicio.


## Paso 6 — Declarar el contrato de datos y validarlo

In [53]:
# Se lee desde BRONCE, no desde el CSV. A partir de aquí, el archivo
# original ya no vuelve a tocarse.
ventas_b = pl.read_parquet(BRONCE / 'ventas_pos' / '**' / '*.parquet')
print(f'Leídas {ventas_b.height:,} filas desde la zona bronce')

CONTRATO_VENTAS = {
    'id_venta': pl.Int64,
    'fecha': pl.Date,
    'id_local': pl.Int64,
    'id_producto': pl.Int64,
    'cantidad': pl.Int64,
    'importe': pl.Float64,
    'canal': pl.String,
    'medio_pago': pl.String,
}

Leídas 200,600 filas desde la zona bronce


In [54]:
def validar_contrato(df, contrato, nombre):
    """Verifica presencia de columnas y compatibilidad de tipos."""
    faltantes = [c for c in contrato if c not in df.columns]
    nuevas = [c for c in df.columns
        if c not in contrato and not c.startswith('_')]
    tipos = [f'{c}: esperado {t}, recibido {df.schema[c]}'
        for c, t in contrato.items()
        if c in df.columns and df.schema[c] != t]

    if nuevas:
        print(f' AVISO — deriva en {nombre}: columnas nuevas {nuevas}')

    if faltantes or tipos:
        raise ValueError(f'Contrato incumplido en {nombre}. '
                         f'Faltantes: {faltantes}. Tipos: {tipos}')
    print(f' Contrato OK en {nombre}: '
        f'{df.height:,} filas x {df.width} columnas')
    return True

validar_contrato(ventas_b, CONTRATO_VENTAS, 'ventas_pos')

 Contrato OK en ventas_pos: 200,600 filas x 11 columnas


True

In [55]:
REGLAS = {
    'importe_nulo': pl.col('importe').is_null(),
    'fecha_futura': pl.col('fecha') > pl.lit(date(2026, 8, 31)),
    'cantidad_cero': pl.col('cantidad') == 0,
}

diagnostico = ventas_b.select([
    expr.sum().alias(nombre) for nombre, expr in REGLAS.items()
])

print('Registros que violan cada regla:')
print(diagnostico)

duplicados = ventas_b.select(pl.exclude('^_.*$')).is_duplicated().sum()
print(f'\nFilas involucradas en duplicados exactos: {duplicados:,}')

Registros que violan cada regla:
shape: (1, 3)
┌──────────────┬──────────────┬───────────────┐
│ importe_nulo ┆ fecha_futura ┆ cantidad_cero │
│ ---          ┆ ---          ┆ ---           │
│ u32          ┆ u32          ┆ u32           │
╞══════════════╪══════════════╪═══════════════╡
│ 4012         ┆ 180          ┆ 0             │
└──────────────┴──────────────┴───────────────┘

Filas involucradas en duplicados exactos: 1,200


In [56]:
sospechoso = (REGLAS['importe_nulo']
              | REGLAS['fecha_futura']
              | REGLAS['cantidad_cero'])

cuarentena = ventas_b.filter(sospechoso)
aceptadas = ventas_b.filter(~sospechoso)

cuarentena.write_parquet(CUARENTENA / 'ventas_pos_rechazadas.parquet')

tasa = 100 * cuarentena.height / ventas_b.height
print(f'Cuarentena: {cuarentena.height:,} filas ({tasa:.2f} %)')
print(f'Aceptadas : {aceptadas.height:,} filas')

Cuarentena: 4,189 filas (2.09 %)
Aceptadas : 196,411 filas


**Punto de control 6**

**Obtiene aproximadamente 4 187 filas en cuarentena y 196 413 aceptadas. Observe que 4 010 + 180
no suma 4 187: la diferencia se debe a los duplicados, que multiplican algunas filas defectuosas. En
una celda de texto, explique por qué las devoluciones (cantidad negativa) NO fueron enviadas a
cuarentena pese a ser valores atípicos.**

Las devoluciones tienen cantidad negativa, pero las reglas solo revisan si la cantidad es igual a cero (cantidad_cero), no si es negativa, entonces nunca se marcan como sospechosas. Ademas una devolucion no es un dato con error, es algo normal del negocio, por eso no se manda a cuarentena sino que se deja y se marca con la columna es_devolucion para poder identificarla despues.

## Paso 7 — Normalizar y construir la zona plata

In [57]:
plata = (aceptadas
    # 1) Deduplicar por las columnas de NEGOCIO, no por las de procedencia
    .unique(subset=['id_venta', 'fecha', 'id_local', 'id_producto',
                    'cantidad', 'importe'], keep='first')


    # 2) Normalizar texto y derivar columnas de análisis
    .with_columns([
        pl.col('canal').str.strip_chars().str.to_titlecase().alias('canal'),
        pl.col('medio_pago').str.strip_chars()
        .str.to_titlecase().alias('medio_pago'),
        (pl.col('cantidad') < 0).alias('es_devolucion'),
        pl.col('fecha').dt.year().alias('anio'),
        pl.col('fecha').dt.month().alias('mes'),
    ])

    # 3) Orden explícito de columnas
    .select(['id_venta', 'fecha', 'anio', 'mes', 'id_local', 'id_producto',
             'cantidad', 'importe', 'canal', 'medio_pago', 'es_devolucion'])
)

print('Canales tras normalizar:', sorted(plata['canal'].unique().to_list()))
print('Devoluciones marcadas :', plata['es_devolucion'].sum())
print('Filas en zona plata :', f'{plata.height:,}')

Canales tras normalizar: ['Delivery', 'Presencial', 'Web']
Devoluciones marcadas : 2937
Filas en zona plata : 195,823


In [58]:
destino_plata = PLATA / 'ventas'
plata.write_parquet(destino_plata,
                    partition_by=['anio', 'mes'],
                    compression='zstd')

# Inspeccionar la estructura de directorios generada
carpetas = sorted(p for p in destino_plata.rglob('anio=*/mes=*') if p.is_dir())
print(f'Particiones creadas: {len(carpetas)}')
for c in carpetas[:5]:
    print(' ', c.relative_to(destino_plata))
print(' ...')

Particiones creadas: 12
  anio=2025/mes=10
  anio=2025/mes=11
  anio=2025/mes=12
  anio=2025/mes=9
  anio=2026/mes=1
 ...


**Punto de control 7**

**La columna canal pasó de 12 valores distintos a exactamente 3. Si obtiene más de 3, revise que
aplicó strip_chars antes de to_titlecase: el orden importa, porque un espacio inicial impide que el
cambio de mayúsculas produzca el mismo resultado.**

## Paso 8 — Medir el efecto del formato columnar

In [59]:
def tam_mb(ruta):
    """Tamaño en MB de un archivo o del árbol completo de un directorio."""
    ruta = Path(ruta)
    if ruta.is_file():
        return ruta.stat().st_size / 1e6
    return sum(f.stat().st_size
            for f in ruta.rglob('*') if f.is_file()) / 1e6

mb_csv = tam_mb(DATOS / 'ventas_pos.csv')
mb_pq = tam_mb(destino_plata)

print(f'CSV original : {mb_csv:7.2f} MB')
print(f'Parquet particion.: {mb_pq:7.2f} MB')
print(f'Reducción : {mb_csv / mb_pq:7.1f}x')

CSV original :   10.54 MB
Parquet particion.:    2.00 MB
Reducción :     5.3x


In [60]:
con = duckdb.connect()

# --- Consulta A: sobre Parquet particionado ---
t0 = time.time()
resultado_pq = con.execute(f"""
    SELECT id_local,
    ROUND(SUM(importe), 2) AS venta_total,
    COUNT(*) AS n_tickets
    FROM read_parquet('{destino_plata}/**/*.parquet', hive_partitioning=true)
    WHERE anio = 2026 AND mes = 3
    GROUP BY id_local
    ORDER BY venta_total DESC
    LIMIT 5
""").df()
t_pq = time.time() - t0

# --- Consulta B: sobre el CSV original ---
t0 = time.time()
resultado_csv = con.execute("""
    SELECT id_local,
    ROUND(SUM(importe), 2) AS venta_total,
    COUNT(*) AS n_tickets
    FROM read_csv_auto('datos/ventas_pos.csv')
    WHERE year(CAST(fecha AS DATE)) = 2026
    AND month(CAST(fecha AS DATE)) = 3
    GROUP BY id_local
    ORDER BY venta_total DESC
    LIMIT 5
""").df()
t_csv = time.time() - t0

print(f'Parquet particionado: {t_pq:.3f} s')
print(f'CSV completo : {t_csv:.3f} s')
print(f'Aceleración : {t_csv / t_pq:.1f}x')

resultado_pq

Parquet particionado: 0.011 s
CSV completo : 0.087 s
Aceleración : 8.1x


,id_local,venta_total,n_tickets
0,42,20005.54,171
1,83,18000.65,153
2,81,17818.89,152
3,120,17416.45,151
4,86,17369.00,147


**Punto de control 8**

**Su cuaderno contiene la tabla completa con sus seis valores. En una celda de texto, explique por
qué la aceleración es mayor que la reducción de tamaño. Pista: la consulta filtra por un solo mes y
usa solo tres de las once columnas disponibles.**

Registro de resultados:

| Métrica | Referencia del docente | Su valor |
|---|---|---|
| Tamaño del CSV original | 10,54 MB | 10,54 MB |
| Tamaño del Parquet particionado | 1,98 MB | 2,00 MB |
| Factor de reducción de tamaño | 5,3x | 5,3x |
| Tiempo de consulta sobre Parquet | 0,014 s | 0,012 s |
| Tiempo de consulta sobre CSV | 0,167 s | 0,083 s |
| Factor de aceleración | 11,6x | 7,2x |

La aceleracion es mayor que la reduccion de tamaño porque el parquet no solo pesa menos, sino que ademas la consulta solo usa 3 de las 11 columnas y en un formato columnar se pueden leer solo esas columnas sin abrir el resto. Tambien ayuda que la consulta filtra por un solo mes y como el parquet esta particionado por año y mes, duckdb ni siquiera abre las carpetas de los otros meses. En cambio con el csv se tiene que leer el archivo completo (todas las columnas y todas las fechas) antes de poder filtrar.

## Paso 9 — Consulta analítica de verificación

In [61]:
resumen_canal = con.execute(f"""
    SELECT canal,
    COUNT(*) AS tickets,
    ROUND(SUM(importe), 2) AS importe_total,
    ROUND(AVG(importe), 2) AS ticket_promedio
    FROM read_parquet('{destino_plata}/**/*.parquet', hive_partitioning=true)
    WHERE NOT es_devolucion
    GROUP BY canal
    ORDER BY importe_total DESC
""").df()

resumen_canal

,canal,tickets,importe_total,ticket_promedio
0,Presencial,138670,14748446.99,106.36
1,Delivery,34726,3688078.43,106.21
2,Web,19490,2067639.39,106.09


**Punto de control 9 — cierre del desarrollo guiado**

**Verifique que se cumplen las cuatro condiciones siguientes antes de pasar a la actividad propuesta:
(1) el manifiesto tiene al menos cinco líneas; (2) existe el archivo de cuarentena; (3) la carpeta de
plata tiene 12 particiones; (4) la consulta de verificación devuelve exactamente tres canales.**

# Actividad Propuesta

La gerencia comercial de Farmacéutica Andina sospecha que la demanda de antigripales
responde a la temperatura, y quiere saber en qué regiones esa relación es más fuerte para
ajustar la reposición de inventario antes del próximo invierno. Le pide a usted, como analista,
que use las fuentes ya aterrizadas en su lakehouse para responder.
Al revisar el trabajo anterior, además, el jefe de datos detectó que hay ventas asociadas a
códigos de producto que no existen en el maestro, y quiere saber cuántas son y qué se hizo
con ellas.

**Tarea 1 — ventas con id_producto huérfano**

Se usa una unión anti (how='anti'), que devuelve las filas de ventas que no tienen id_producto en el maestro.

In [62]:
maestro = pl.read_parquet(BRONCE / 'maestro_productos' / '**' / '*.parquet')
locales_m = pl.read_parquet(BRONCE / 'locales' / '**' / '*.parquet')
plata_ventas = pl.read_parquet(f'{destino_plata}/**/*.parquet', hive_partitioning=True)
print(f'Filas en zona plata: {plata_ventas.height:,}')

ventas_huerfanas = plata_ventas.join(
    maestro.select('id_producto'), on='id_producto', how='anti')

codigos_huerfanos = ventas_huerfanas['id_producto'].n_unique()

print(f'Ventas con id_producto huérfano : {ventas_huerfanas.height:,}')
print(f'Códigos de producto distintos sin maestro: {codigos_huerfanos}')

Filas en zona plata: 195,823
Ventas con id_producto huérfano : 341
Códigos de producto distintos sin maestro: 98


**Tarea 2 — tabla enriquecida (plata + maestro de productos + locales)**

Uso union izquierda (how='left') desde plata hacia los maestros, no inner. Como en la tarea 1 se vio que hay 341 ventas cuyo id_producto no esta en el maestro, si uso inner esas filas se perderian sin darme cuenta y bajaria el total de ventas. Con left se quedan igual pero con categoria en null, asi se pueden seguir contando.

In [63]:
enriquecida = (plata_ventas
    .join(maestro.select(['id_producto', 'categoria']), on='id_producto', how='left')
    .join(locales_m.select(['id_local', 'region']), on='id_local', how='left')
)

print(f'Filas en tabla enriquecida : {enriquecida.height:,}')
print(f'Filas con categoria nula (ventas huérfanas): {enriquecida["categoria"].null_count():,}')
enriquecida.head(5)

Filas en tabla enriquecida : 195,823
Filas con categoria nula (ventas huérfanas): 341


id_venta,fecha,anio,mes,id_local,id_producto,cantidad,importe,canal,medio_pago,es_devolucion,categoria,region
i64,date,i32,i8,i64,i64,i64,f64,str,str,bool,str,str
50375,2025-10-10,2025,10,20,1489,4,301.16,"""Presencial""","""Tarjeta""",false,"""Dermatológicos""","""Arequipa"""
179703,2025-10-20,2025,10,79,1612,3,101.99,"""Presencial""","""Efectivo""",false,"""Cuidado infantil""","""Junín"""
194143,2025-10-31,2025,10,64,1138,6,111.88,"""Presencial""","""Efectivo""",false,"""Analgésicos""","""Áncash"""
88391,2025-10-06,2025,10,74,1630,2,14.0,"""Web""","""Efectivo""",false,"""Higiene personal""","""Cusco"""
65,2025-10-10,2025,10,7,1793,5,66.54,"""Presencial""","""Billetera""",false,"""Analgésicos""","""Cusco"""


**Tarea 3 — cruce con el clima por región y fecha**

La clave es compuesta, on=['region', 'fecha']. La region viene del maestro de locales, no de la tabla de ventas original.

In [64]:
clima_b = pl.read_parquet(BRONCE / 'clima_diario' / '**' / '*.parquet')

enriquecida_clima = enriquecida.join(
    clima_b.select(['region', 'fecha', 'temperatura_c']),
    on=['region', 'fecha'],
    how='left'
)

sin_temp = enriquecida_clima['temperatura_c'].null_count()
print(f'Filas en tabla enriquecida + clima: {enriquecida_clima.height:,}')
print(f'Filas sin temperatura asociada : {sin_temp:,}')

Filas en tabla enriquecida + clima: 195,823
Filas sin temperatura asociada : 0


Da 0 filas sin temperatura. Tiene sentido porque las 8 regiones del maestro de locales son las mismas 8 que estan en clima_diario, y el rango de fechas del clima cubre todo el rango de las ventas. Si hubiera quedado algo sin temperatura, seria por una region mal escrita entre los dos archivos o por una fecha de venta fuera del periodo que cubre el clima.

**Tarea 4 — correlación temperatura-cantidad, Antigripales por región**

In [65]:
antigripales = enriquecida_clima.filter(
    (pl.col('categoria') == 'Antigripales') & (~pl.col('es_devolucion'))
)

correlacion_region = (antigripales
    .group_by('region')
    .agg(pl.corr('temperatura_c', 'cantidad').alias('correlacion'))
    .sort('correlacion')
)

print(f'Filas de Antigripales sin devoluciones: {antigripales.height:,}')
correlacion_region

Filas de Antigripales sin devoluciones: 34,634


region,correlacion
str,f64
"""Áncash""",-0.370439
"""Arequipa""",-0.355266
"""Lambayeque""",-0.349782
"""Lima""",-0.344865
"""Piura""",-0.339196
"""Junín""",-0.337174
"""La Libertad""",-0.330735
"""Cusco""",-0.330669


**Tarea 5 — ficha de fuente (Anexo B) y conclusión**

Ficha de fuente para clima_diario.jsonl:

| Campo | Contenido |
|---|---|
| Nombre de la fuente | clima_diario.jsonl |
| Entidad responsable | Servicio meteorológico (simulado) |
| Tipo (origen / estructura) | Externa, semiestructurada, JSON Lines anidado |
| Modo de acceso y patrón de ingesta | Archivo entregado por el docente, se lee completo y se aterriza en bronce cada vez |
| Período cubierto y granularidad | 2025-09-01 a 2026-08-31, diario por región |
| Frecuencia de actualización | Diaria |
| Fecha de obtención | 2026-09-01 |
| Volumen (filas y tamaño) | 2920 registros, ~0,47 MB |
| Licencia o base de licitud | Dato simulado con fines academicos |
| ¿Contiene datos personales? ¿Cuáles? | No, son mediciones de clima por región |
| Tratamiento de anonimización aplicado | No aplica |
| Problemas de calidad detectados | Deriva de esquema: a mitad del período el campo temperatura_c pasó a llamarse temp_c y se agregó indice_uv, sin aviso. Se resolvió uniendo ambos nombres en una sola columna y guardando la version (v1/v2) en _esquema |
| Decisión: se incorpora o se descarta, y por qué | Se incorpora, es la unica forma de cruzar clima con ventas por region y fecha, y el cruce no dejo filas sin temperatura |

Conclusión para la gerencia:

Se ve una correlacion negativa entre temperatura y venta de antigripales en las 8 regiones (entre -0,33 y -0,37), o sea que a menor temperatura se vende mas. Es un poco mas marcada en Áncash, Arequipa y Lambayeque, asi que ahi convendria reforzar el inventario antes del invierno. Igual hay que tener cuidado, correlacion no es causalidad: puede que la temperatura solo este reflejando la temporada de gripes y no sea la causa directa.